# Imports

In [4]:
from datasets import load_dataset
import pandas as pd
import random
import re
import json
import os
from dotenv import load_dotenv

load_dotenv()  # laedt GEMINI_API_KEY aus der .env in os.environ

True

# Daten anschauen

In [ ]:
dataset = load_dataset("jfrei/GPTNERMED", trust_remote_code=True)

: 

: 

In [ ]:
print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 7876
    })
    validation: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 984
    })
    test: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 985
    })
})
{'sentence': Value(dtype='string', id=None), 'ner_labels': Sequence(feature={'ner_class': ClassLabel(names=['Medikation', 'Dosis', 'Diagnose'], id=None), 'start': Value(dtype='int32', id=None), 'stop': Value(dtype='int32', id=None)}, length=-1, id=None)}
{'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}


: 

: 

In [ ]:
example = dataset["train"][0]
text = example["sentence"]
label = example["ner_labels"]
print(text)
print(label)

0,4 Diuretika 0,25 1x/die
{'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}


: 

: 

In [ ]:
for i in range(5):
    print(dataset["train"][i])

{'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}
{'sentence': '0,5mg Sildenafil bei erectilen Dysfunktion', 'ner_labels': {'ner_class': [1, 0, 2], 'start': [0, 6, 21], 'stop': [5, 16, 42]}}
{'sentence': '1.000 mg Phenprobenon über 24h, 200 mg Phenprobenon über 4 h.', 'ner_labels': {'ner_class': [1, 0, 1, 0], 'start': [0, 9, 32, 39], 'stop': [5, 21, 35, 51]}}
{'sentence': '10 Ranitidine', 'ner_labels': {'ner_class': [1, 0], 'start': [0, 3], 'stop': [2, 13]}}
{'sentence': '10 mg Enalapril 40mg Metoprolol 25mg', 'ner_labels': {'ner_class': [1, 0, 1, 0, 1], 'start': [0, 6, 16, 21, 32], 'stop': [5, 15, 20, 31, 36]}}


: 

: 

# LLM Generated Data (synthetic data)
- Paper probleme: Halluzinierte Begriffe und strukturelle Homogenität
- Idee: Echt medikationsnamen etc aus offiziellen Listen ziehen
- dazu: lables per matching erstellen, da LLMs teilweise schwierigkeiten damit haben

Hier: Versuche erstmal einen Satz zu erzeugung - genutzt dafür ein anteil von Beispiel Medikamenten und Diagnosen und Sätzen

In [ ]:
FALLBACK_MEDS = ["Ramipril", "Metformin", "Ibuprofen", "Pantoprazol",
                 "Amoxicillin", "Bisoprolol", "Simvastatin", "L-Thyroxin"]
FALLBACK_DIAGS = ["arterielle Hypertonie", "Diabetes mellitus Typ 2", "Migraene",
                  "Refluxoesophagitis", "Pneumonie", "Hypothyreose",
                  "Vorhofflimmern", "Asthma bronchiale"]
DOSE_UNITS = ["mg", "ug", "g", "ml", "IE"]
SHAPES = [
    ("Medikation", "Dosis"),
    ("Medikation", "Dosis", "Diagnose"),
    ("Diagnose",),
    ("Medikation", "Diagnose"),
    ("Medikation", "Dosis", "Diagnose", "Diagnose"),
]

# original 12 sentences from prompt used to generate the GPTNERMED dataset
ORIGINAL_SENTENCES = [
    '<s>Zur weiteren Bekämpfung des <class="Diagnose">Juckreiz</class> wird die Einnahme von täglich <class="Dosis">100mg</class> <class="Medikation">Cortison</class> empfohlen.</s>',
    '<s>Bei wiederkehrender Infektion wie einer <class="Diagnose">Sepsis</class> oder schweren <class="Diagnose">Pnseumonien</class> wird eine Überwachung erforderlich sein.</s>',
    '<s><class="Medikation">Valsartan</class>/<class="Medikation">HCT</class> <class="Dosis">160</class>/<class="Dosis">12,5 mg</class> 1-0-0</s>',
    '<s><class="Medikation">Pantoprazol</class> <class="Dosis">40 mg</class> p.o.</s>',
    '<s>Die feingewebliche histopathologische Untersuchung ergab den Befund einer <class="Diagnose">Metastase</class> des bekannten malignen <class="Diagnose">Melanoms</class>.</s>',
    '<s><class="Diagnose">Diabetes Typ 2</class>-Patienten müssen regelmäßig <class="Medikation">Insulin</class> (mindestens mit <class="Dosis">12ml</class> dosiert) spritzen.</s>',
    '<s>Ich nehme <class="Medikation">Antibiotika</class> seit Tagen. Seitdem ist die <class="Diagnose">Mandelentzündung</class> deutlich besser geworden.</s>',
    '<s>Entlassung: <class="Dosis">40mg</class> <class="Medikation">Lidocain</class> wegen <class="Diagnose">Kopfschmerzen</class></s>',
    '<s>Zusammenfassende D: Zervix-PE bei 11 und 2 Uhr mit ausgeprägter <class="Diagnose">chronisch-florider Zervizitis</class>.</s>',
    '<s>Die Verschreibung von <class="Medikation">Hämatokrin</class> <class="Dosis">43mg</class> war unnötig.</s>',
    '<s>Der Patient klagt über <class="Diagnose">Karditiden</class> und nimmt täglich <class="Medikation">Nifedipin</class> ein.</s>',
    '<s>D: PE-Material der Portio bei 1 Uhr mit Nachweis einer schwergradigen <class="Diagnose">squamösen intraepithelialen Läsion</class> (<class="Diagnose">HSIL</class>; hier noch <class="Diagnose">CIN II</class>).</s>',
]

: 

: 

Beispiel Dosis

In [ ]:
def make_dosage(rng):
    val = rng.choice([2.5, 5, 10, 12.5, 20, 25, 40, 50, 100, 250, 500])
    unit = rng.choice(DOSE_UNITS)
    
    space_prob = rng.random()
    if (space_prob < 0.7):
        text = f"{val}{unit}"
    else:
        text = f"{val} {unit}"

    return text.replace(".", ",")    

rng = random.Random(42)
print(make_dosage(rng))

500mg


: 

: 

Beispiel Entität: Medikation, Diagnose, Dosis

In [ ]:
def sample_entities(rng, medication, diagnoses):
    entities = []
    for word in rng.choice(SHAPES):
        if word == "Medikation":
            entities.append({"text": rng.choice(medication), "label": "Medikation"})
        elif word == "Dosis":
            entities.append({"text": make_dosage(rng), "label": "Dosis"})
        elif word == "Diagnose":
            entities.append({"text": rng.choice(diagnoses), "label": "Diagnose"})
    
    return entities

rng = random.Random(42)
for _ in range(3):
    print(sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS))

[{'text': 'Ramipril', 'label': 'Medikation'}, {'text': '20ug', 'label': 'Dosis'}]
[{'text': 'Metformin', 'label': 'Medikation'}, {'text': '250ml', 'label': 'Dosis'}]
[{'text': 'Pantoprazol', 'label': 'Medikation'}, {'text': '12,5IE', 'label': 'Dosis'}]


: 

: 

## Gemini API
- test API aufruf

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Antworte mit genau einem Satz: Was ist maschinelles Lernen?",
)

print(response.text)

Maschinelles Lernen ist ein Teilbereich der Künstlichen Intelligenz, bei dem Computersysteme anhand von Daten eigenständig Muster erkennen und statistische Modelle entwickeln, um Vorhersagen oder Entscheidungen zu treffen, ohne explizit für jede Aufgabe programmiert worden zu sein.


: 

: 

# Erste genierieung text

In [ ]:
SYSTEM_PROMT = (
    "Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das "
    "Training eines NER-Modells. Regeln:\n"
    "1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.\n"
    "2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.\n"
    "3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.\n"
    "4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar."
)

"""
{"text": "0,5mg Digoxin 0,25mg Lanatoside", "label": [[0, 5, "Dosis"], [6, 13, "Medikation"], [14, 20, "Dosis"], [21, 31, "Medikation"]]}
{"text": "0,5mg Sildenafil bei erectilen Dysfunktion", "label": [[0, 5, "Dosis"], [6, 16, "Medikation"], [21, 42, "Diagnose"]]}
"""
EXAMPLE_SAMPLES = [
    '{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}',
    '{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}',

]

: 

: 

In [ ]:
def build_task(entities):
    lines = ["Begriffe:"]
    for ent in entities:
        lines.append(f"- {ent['label']}: {ent['text']}")
    return "\n".join(lines)

print(build_task(sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS)))

Begriffe:
- Medikation: Pantoprazol
- Dosis: 500IE
- Diagnose: Asthma bronchiale
- Diagnose: Pneumonie


: 

: 

In [ ]:
def build_format():
    lines = ["Format:"]
    lines.append('{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}')
    return "\n".join(lines)

print(build_format())

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}


: 

: 

In [ ]:
def build_llm_prompt(rng, medication, diagnoses, entities):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append("Beispiel Sätze:\n")
    for sentence in EXAMPLE_SAMPLES:
        message.append(f"{sentence}\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS)  
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das Training eines NER-Modells. Regeln:
1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.
2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.
3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.
4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar.

User:
Begriffe:
- Medikation: Ramipril
- Dosis: 20ug

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}

Beispiel Sätze:
{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}
{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}



: 

: 

In [ ]:
from google import genai

def call_LLM(model, message):

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    print(response.text)
    return response.text

model = "gemini-3.1-flash-lite"
message = build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities)
generated_sample = call_LLM(model, message)

{"text": "Der Patient erhält täglich 20ug Ramipril.", "entities": [{"text": "20ug", "label": "Dosis"}, {"text": "Ramipril", "label": "Medikation"}]}


: 

: 

check

In [ ]:
print(f"LLM Prompt: \n {message}")
print(f"Entities: {entities}")
print("\n")
print(f"LLM Output: \n {generated_sample}")

LLM Prompt: 
 System: Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das Training eines NER-Modells. Regeln:
1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.
2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.
3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.
4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar.

User:
Begriffe:
- Medikation: Ramipril
- Dosis: 20ug

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}

Beispiel Sätze:
{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}
{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}



: 

: 

# Baue daraus Huggingface Datensatz form

In [ ]:
#form anschauen
example = dataset["train"][0]
print(f"Form die ich habe: \n {generated_sample}")

print(f"Form die ich möchte: \n {example}")

Form die ich habe: 
 {"text": "Der Patient erhält täglich 20ug Ramipril.", "entities": [{"text": "20ug", "label": "Dosis"}, {"text": "Ramipril", "label": "Medikation"}]}
Form die ich möchte: 
 {'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}


: 

: 

In [ ]:
def get_label_id(label):
    label_mapping = {
        "Medikation": 0,
        "Dosis": 1,
        "Diagnose": 2
    }
    return label_mapping.get(label)

def find_spans(sentence, entities):
    spans = []

    for ent in entities:
        ent_lower = ent["text"].lower()
        sentence = sentence.lower()
        
        start = sentence.find(ent_lower)
        stop = start + len(ent_lower)
        
        spans.append({
            "start": start,
            "stop": stop,
            "label": ent["label"]
        })

    spans.sort(key=lambda span: span["start"])

    return {
        "ner_class": [get_label_id(span["label"]) for span in spans],
        "start": [span["start"] for span in spans],
        "stop": [span["stop"] for span in spans]
    }

# Punkt 1: erst JSON parsen, dann die Begriffe im Satz-Text matchen
parsed = json.loads(generated_sample)
spans = find_spans(parsed["text"], entities)

print(parsed["text"])
print(entities)
print(spans)

NameError: name 'json' is not defined

: 

: 

In [ ]:
import json

def build_dataset_form(sentence, spans):
    sentence = json.loads(sentence)
    return {'sentence': sentence["text"], 'ner_labels': spans}

print(build_dataset_form(generated_sample, spans))

{'sentence': 'Der Patient erhält täglich 20ug Ramipril.', 'ner_labels': {'ner_class': [1, 0], 'start': [27, 32], 'stop': [31, 40]}}


: 

: 

# Nun teste aus Medizinischen tabellen zu nehemn

In [ ]:
import pandas as pd

AlphaIDSE = pd.read_csv(
    "/Users/wielandcremer/Dokumente/Uni/6_Semester/"
    "information_extraction_german_medical_data/"
    "meaningful_modification/data/medical_information/"
    "Alpha-ID-SE.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str
)

whoACTddd = pd.read_csv("/Users/wielandcremer/Documents/Uni/6_Semester/information_extraction_german_medical_data/meaningful_modification/data/medical_information/who_atc_ddd.csv")

whoACTddd = whoACTddd.dropna(subset="ddd")
whoACTddd = whoACTddd.dropna(subset="uom")

: 

: 

In [ ]:
MEDS = (whoACTddd["atc_name"].astype(str).to_list())
DOSE_UNITS = (whoACTddd["uom"].astype(str).to_list())
DOSE_VAL = (whoACTddd["ddd"].astype(str).to_list())
DIAGS = (AlphaIDSE["Diagnose"].astype(str).to_list())

: 

: 

In [ ]:
SHAPES = [
    ("Medikation", "Dosis"),
    ("Medikation", "Dosis", "Diagnose"),
    ("Diagnose",),
    ("Medikation", "Diagnose"),
    ("Medikation", "Dosis", "Diagnose", "Diagnose"),
]

: 

: 

In [ ]:
print(MEDS)

['sodium fluoride', 'olaflur', 'hydrogen peroxide', 'chlorhexidine', 'amphotericin B', 'polynoxylin', 'domiphen', 'oxyquinoline', 'miconazole', 'natamycin', 'minocycline', 'magnesium hydroxide', 'algeldrate', 'dihydroxialumini sodium carbonate', 'calcium silicate', 'cimetidine', 'cimetidine', 'ranitidine', 'ranitidine', 'famotidine', 'famotidine', 'nizatidine', 'nizatidine', 'roxatidine', 'ranitidine bismuth citrate', 'lafutidine', 'misoprostol', 'enprostil', 'omeprazole', 'omeprazole', 'pantoprazole', 'pantoprazole', 'lansoprazole', 'rabeprazole', 'esomeprazole', 'esomeprazole', 'dexlansoprazole', 'carbenoxolone', 'sucralfate', 'pirenzepine', 'pirenzepine', 'bismuth subcitrate', 'proglumide', 'rebamipide', 'oxyphencyclimine', 'camylofin', 'camylofin', 'mebeverine', 'trimebutine', 'dicycloverine', 'dicycloverine', 'benzilone', 'glycopyrronium bromide', 'glycopyrronium bromide', 'glycopyrronium bromide', 'oxyphenonium', 'penthienate', 'propantheline', 'propantheline', 'methantheline', '

: 

: 

In [ ]:
print(DOSE_UNITS)

['mg', 'mg', 'mg', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mcg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'g', 'mg', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'MU', 'g', 'MU', 'g', 'g', 'g', 'g', 'MU', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'g', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 

: 

: 

## Code von oben für neuen fall angepasst (später so im script)

In [ ]:
def make_dosage(rng, values, units):
    val = rng.choice(values)
    unit = rng.choice(units)
    
    space_prob = rng.random()
    if (space_prob < 0.7):
        text = f"{val}{unit}"
    else:
        text = f"{val} {unit}"

    return text.replace(".", ",")    

rng = random.Random(42)
print(make_dosage(rng, DOSE_VAL, DOSE_UNITS))

1,4 mg


: 

: 

In [ ]:
def sample_entities(rng, medication, diagnoses, dose_values, dose_units):
    entities = []
    for word in rng.choice(SHAPES):
        if word == "Medikation":
            entities.append({"text": rng.choice(medication), "label": "Medikation"})
        elif word == "Dosis":
            entities.append({"text": make_dosage(rng,dose_values, dose_units), "label": "Dosis"})
        elif word == "Diagnose":
            entities.append({"text": rng.choice(diagnoses), "label": "Diagnose"})
    
    return entities

rng = random.Random(42)
for _ in range(3):
    print(sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS))

[{'text': 'ondansetron', 'label': 'Medikation'}, {'text': '1,0mg', 'label': 'Dosis'}]
[{'text': 'hydroxychloroquine', 'label': 'Medikation'}, {'text': '7,5g', 'label': 'Dosis'}]
[{'text': 'tirofiban', 'label': 'Medikation'}, {'text': '10,0mg', 'label': 'Dosis'}]


: 

: 

In [ ]:
"""
SYSTEM_PROMT = (
    "Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das "
    "Training eines NER-Modells. Regeln:\n"
    "1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.\n"
    "2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.\n"
    "3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.\n"
    "4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar."
)
"""
SYSTEM_PROMT = (
    
    "Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells."

    "Verbindliche Regeln:"

    "1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.\n"
    "2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.\n"
    "3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.\n"
    "4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.\n"
    "5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.\n"
    "6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.\n"
    "7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.\n"
    "8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.\n"
    "9. Die Einträge in „entities“ müssen in derselben Reihenfolge stehen, in der die Begriffe im Satz vorkommen.\n"
    "10. Verwende ausschließlich diese Labels: „Medikation“, „Dosis“ und „Diagnose“.\n"
    "11. Verwende keine Entität, die nicht in den Eingabebegriffen enthalten ist.\n"
    "12. Prüfe vor der Ausgabe intern:\n"
    "- Sind alle Begriffe exakt einmal enthalten?\n"
    "- Wurden keine medizinischen Informationen ergänzt?\n"
    "- Stimmen Text und Labels exakt überein?\n"
    "- Ist das JSON syntaktisch gültig?\n"
    "13. Antworte ausschließlich mit einem einzelnen gültigen JSON-Objekt. Kein Markdown, keine Erklärung und kein zusätzlicher Text.\n"
)

"""
{"text": "0,5mg Digoxin 0,25mg Lanatoside", "label": [[0, 5, "Dosis"], [6, 13, "Medikation"], [14, 20, "Dosis"], [21, 31, "Medikation"]]}
{"text": "0,5mg Sildenafil bei erectilen Dysfunktion", "label": [[0, 5, "Dosis"], [6, 16, "Medikation"], [21, 42, "Diagnose"]]}
"""
EXAMPLE_SAMPLES = [
    '{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}',
    '{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}',

]

: 

: 

hier noch später testen, ob vielleicht mehr oder andere Sätze immer als Beispiel genutzt werden. Zum Beispiel könnte man hier zufällige sätze aus dem Datensatz nehmen und als Examples geben

In [ ]:
LASS_RE = re.compile(r'<class="([^"]+)">(.*?)</class>')

def parse_original_sentence(tagged):
    inner = tagged.replace("<s>", "").replace("</s>", "")

    entities = [
        {"text": match.group(2), "label": match.group(1)}
        for match in CLASS_RE.finditer(inner)
    ]
    text = CLASS_RE.sub(r"\2", inner)   # Tags entfernen, nur Inhalt behalten

    return {"text": text, "entities": entities}


EXAMPLE_POOL = [parse_original_sentence(sentence) for sentence in ORIGINAL_SENTENCES]


def build_examples(rng, pool=EXAMPLE_POOL, num_examples=2):
    chosen = rng.sample(pool, num_examples)
    lines = ["Beispiel Sätze:"]
    for example in chosen:
        lines.append(json.dumps(example, ensure_ascii=False))
    return "\n".join(lines)


# check
for example in EXAMPLE_POOL[:3]:
    print(json.dumps(example, ensure_ascii=False))

print("\n--- zwei random Beispiele ---")
print(build_examples(random.Random(42)))

NameError: name 'ORIGINAL_SENTENCES' is not defined

: 

In [ ]:
def build_task(entities):
    lines = ["Begriffe:"]
    for ent in entities:
        lines.append(f"- {ent['label']}: {ent['text']}")
    return "\n".join(lines)


rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
task = build_task(entities)
print(task)

Begriffe:
- Medikation: ondansetron
- Dosis: 1,0mg


: 

: 

In [ ]:
def build_format():
    lines = ["Format:"]
    lines.append('{"text": "...", "label": [[start, stop, "Medikation|Dosis|Diagnose"], ...]}')
    return "\n".join(lines)

print(build_format())

Format:
{"text": "...", "label": [[start, stop, "Medikation|Dosis|Diagnose"], ...]}


: 

: 

In [ ]:
def build_llm_prompt(rng, medication, diagnoses, entities, example_pool=EXAMPLE_POOL, num_examples=2):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append(build_examples(rng, example_pool, num_examples))
    message.append("\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.Verbindliche Regeln:1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.
2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.
3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.
4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.
5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.
6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.
7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.
8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.
9. Die Einträge in „entities“ müsse

: 

: 

In [ ]:
from google import genai

def call_LLM(model, message):
    
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    print(response.text)
    return response.text

model = "gemini-3.1-flash-lite"

for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities)
    generated_sample = call_LLM(model, message)

{"text": "Patient mit HIV-Krankheit mit Pneumocystosis und Ektoper Aldosteron-produzierender Tumor des Hodens erhält warfarin in einer Dosis von 1,2mg.", "label": [[12, 44, "Diagnose"], [49, 94, "Diagnose"], [99, 106, "Medikation"], [125, 129, "Dosis"]]}
{"text": "Patient mit Andersen-Tawil-Syndrom sowie Syndrom der Pulmonalklappenagenesie mit Fallot-Tetralogie und fehlendem Ductus arteriosus erhält tropisetron 0,24 g.", "label": [[12, 33, "Diagnose"], [39, 116, "Diagnose"], [124, 135, "Medikation"], [136, 142, "Dosis"]]}
{
  "text": "Die Therapie mit cefotiam in einer Dosis von 0,8mg behandelt das Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom sowie das Polyendokrines Polyneuropathie-Syndrom.",
  "label": [
    [14, 22, "Medikation"],
    [38, 43, "Dosis"],
    [63, 114, "Diagnose"],
    [125, 161, "Diagnose"]
  ]
}
{"text": "Der Patient leidet unter einer HIV-Krankheit mit progressivem asymptomatischen Kaposi-Sarkom.", "label": [[24, 78, "Diagnose"]]}
{"text": "Die Therap

: 

: 